# Small Models, Same Rules — clean Colab run (checkpointed)

Runs the full audit grid end to end, with **within-model checkpointing** so a crash or Colab pre-emption never recomputes finished work.

What it does:
1. Sets up the registered code, dependencies, and gated-model access.
2. Runs the grid **one model at a time** — each model is generated once and scored by all three judges (keyword, Llama-Guard-3-1B, HarmBench classifier) in one pass.
3. Writes results **directly to Google Drive**, flushing a checkpoint every 25 prompts. If anything dies, just re-run the cell: it reloads the checkpoint and **skips every prompt already done**.
4. Merges the four models and runs the analysis (tables, confidence intervals, figures).

**Before you start:** use a **freshly rotated** Hugging Face token (the earlier one was exposed), and make sure your pre-registration tag is already public (it is).

---
### Where does data go? (read once)
- **Model weights + datasets** download to the Hugging Face cache `~/.cache/huggingface/` (`hub/` for the 6 models, `datasets/` for HarmBench + XSTest). This lives on the Colab VM and is **wiped on every runtime reset** — a fresh session re-downloads them (the 13B HarmBench judge is the big one, ~26 GB). Nothing you need to keep lives here.
- **Results** are written to `--output_dir`, which we point **straight at Google Drive**. That is the only thing that must survive, and it does — including the partial checkpoint.

### Why this won't waste your compute hours
- **Resume is on by default.** A crash at prompt 605/800 costs the last <25 prompts, not the whole model.
- Greedy decoding (`temperature 0`) means a resumed run is **bit-identical** to an uninterrupted one — free recovery, no scientific cost.
- Note on parallelism (below): running models in separate sessions saves **wall-clock, not compute units** — 4 GPUs for 1 h still bills the same as 1 GPU for 4 h. The thing that saves *units* is not recomputing, which the checkpoint handles.

## Step 0 — Pick a GPU

Runtime → Change runtime type → **L4 GPU** (recommended) or **A100**. The T4 throttled and crashed on the long runs. L4 has 22 GB (no out-of-memory on the 13B judge) and is much faster. The next cell shows which GPU you got.

In [ ]:
!nvidia-smi

## Step 1 — Clone the registered code

Clones your repo at the pre-registered commit `prereg-v1` (the exact frozen protocol). The `rm -rf` makes this safe to re-run after a reset, and absolute paths mean the working directory is always correct.

> The within-model checkpoint/resume code is an operational change committed **after** `prereg-v1`. It does not alter the protocol or the decoding logic (greedy → bit-identical outputs); it only persists progress and skips finished prompts. If you want the resume feature, check out the branch that contains it instead of the bare tag — set `CODE_REF` below.

In [ ]:
REPO_URL = "https://github.com/shubmittal/Jailbreak-Robustness-Small-LLMs.git"
CODE_REF = "main"   # use the commit/branch that has the checkpoint feature; or "prereg-v1" for the bare frozen code
!rm -rf /content/repo
!git clone $REPO_URL /content/repo
%cd /content/repo
!git checkout $CODE_REF
!git log -1 --oneline

## Step 2 — Install dependencies

Takes a couple of minutes.

In [ ]:
!pip install -q -r requirements.txt

## Step 3 — Hugging Face login

First, on huggingface.co, **accept the license** on each gated model page (signed in as the same account your token belongs to):
- `meta-llama/Llama-3.2-3B-Instruct`
- `meta-llama/Llama-Guard-3-1B`
- `google/gemma-2-2b-it`  ← the one missed last time (Google's license is separate from Meta's)

Then run this and paste a **freshly rotated** token. (Do **not** hardcode the token in a cell — the old notebook had it in plaintext and that token is now compromised; rotate it at huggingface.co/settings/tokens.)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Step 4 — Mount Google Drive

Every result and every checkpoint is written **here**, so nothing is lost if the runtime resets. `RUN_DIR` is the single place all outputs live.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/jailbreak_results/run'
import os
os.makedirs(RUN_DIR, exist_ok=True)
print('outputs ->', RUN_DIR)

## Step 5 — Smoke test (~2 min) — canary on Gemma

Runs 2 prompts through **Gemma** specifically, because Gemma is the model whose license failed last time. If this cell loads Gemma and finishes, the license is accepted and the long run will work. It also exercises the checkpoint code on the real path. Throwaway output.

In [ ]:
!python 05_experiment.py --models google/gemma-2-2b-it --model-revisions 299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8 --judges keyword --n 2 --max_new_tokens 16 --no_plot --output_dir ./results/smoke

## Step 6 — Run the grid, one model per cell

Each cell generates 800 completions (200 HarmBench + 200 XSTest × 2 prompt conditions), scores them with all three judges, and streams results to its own Drive folder, checkpointing every 25 prompts.

**If one crashes, just re-run that same cell** — it reloads the checkpoint from Drive and continues from where it stopped (you'll see a `[resume] loaded …` line). Each model is roughly 30–60 min on an L4.

> **Run them in parallel (optional, saves wall-clock):** open this notebook in 2–4 Colab sessions, run Steps 0–4 in each, then run **one** model cell per session. Each writes to a different `RUN_DIR/<model>` folder, so they never collide. Merge (Step 7) once all four folders exist. This needs Colab Pro (and Pro+ for 3–4 at once) and bills units 4× faster — same total units, less waiting.

### ⚡ Run in parallel — checklist (to cut the waiting)

Same total compute units, but ~4× less wall-clock, so you're less likely to drift away mid-run. **You don't have to track who finished — re-running everything is always safe** (done models skip instantly, partial ones resume).

1. **Open this notebook in N browser tabs.** The same notebook in multiple tabs each gets its own runtime. N = 2 on Colab Pro, up to ~4 on Pro+.
2. **In each tab:** Runtime → Change runtime type → **L4**, then run **Steps 0–4** (GPU check, clone, install, HF login, mount Drive). All tabs share the same Drive, so `RUN_DIR` is identical everywhere — that's fine.
3. **Assign one model per tab:** run **only** the Llama cell in tab 1, **only** Phi-3 in tab 2, Qwen in tab 3, Gemma in tab 4. Each writes to its own `RUN_DIR/<model>` folder, so they never collide.
4. **If a tab dies:** re-run that tab's model cell — it resumes from the Drive checkpoint. Forgot which tab had which model? Re-run all four cells in one tab; finished models are skipped in seconds.
5. **When all four `RUN_DIR/<model>` folders exist,** run **Steps 7–9** in any one tab to merge + analyze.

> Fewer tabs is fine: 2 tabs (2 models each, back-to-back) already halves the wait, and every cell still resumes. Sequential in one tab also works — just slower, not riskier.

In [ ]:
!python 05_experiment.py --backend transformers --models meta-llama/Llama-3.2-3B-Instruct --model-revisions 0cb88a4f764b7a12671c53f0838cd831a0843b95 --judges keyword,llamaguard,harmbench --harmbench-cls-size large --n 200 --defense primary --check-prompt-hash 7adca1d95a6759f1eeab9e4ffe45aa5e33ea82a6fe2c57f74c819cd918cf0beb --checkpoint-every 25 --output_dir $RUN_DIR/llama
print('Llama-3.2-3B done + on Drive')

In [ ]:
!python 05_experiment.py --backend transformers --models microsoft/Phi-3-mini-4k-instruct --model-revisions f39ac1d28e925b323eae81227eaba4464caced4e --judges keyword,llamaguard,harmbench --harmbench-cls-size large --n 200 --defense primary --check-prompt-hash 7adca1d95a6759f1eeab9e4ffe45aa5e33ea82a6fe2c57f74c819cd918cf0beb --checkpoint-every 25 --output_dir $RUN_DIR/phi3
print('Phi-3-mini done + on Drive')

In [ ]:
!python 05_experiment.py --backend transformers --models Qwen/Qwen2.5-3B-Instruct --model-revisions aa8e72537993ba99e69dfaafa59ed015b17504d1 --judges keyword,llamaguard,harmbench --harmbench-cls-size large --n 200 --defense primary --check-prompt-hash 7adca1d95a6759f1eeab9e4ffe45aa5e33ea82a6fe2c57f74c819cd918cf0beb --checkpoint-every 25 --output_dir $RUN_DIR/qwen
print('Qwen2.5-3B done + on Drive')

In [ ]:
!python 05_experiment.py --backend transformers --models google/gemma-2-2b-it --model-revisions 299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8 --judges keyword,llamaguard,harmbench --harmbench-cls-size large --n 200 --defense primary --check-prompt-hash 7adca1d95a6759f1eeab9e4ffe45aa5e33ea82a6fe2c57f74c819cd918cf0beb --checkpoint-every 25 --output_dir $RUN_DIR/gemma
print('Gemma-2-2B done + on Drive')

## Step 7 — Merge the four models into one results.csv

In [ ]:
import pandas as pd, os
parts = []
for name in ['llama', 'phi3', 'qwen', 'gemma']:
    p = f'{RUN_DIR}/{name}/results.csv'
    if os.path.exists(p):
        d = pd.read_csv(p); parts.append(d); print(name, len(d), 'rows')
    else:
        print('MISSING:', p)
combined = pd.concat(parts, ignore_index=True)
os.makedirs(f'{RUN_DIR}/combined', exist_ok=True)
combined.to_csv(f'{RUN_DIR}/combined/results.csv', index=False)
print('TOTAL rows:', len(combined), '(expect 9600 = 4 models x 800 prompts x 3 judges)')
print(combined['model'].value_counts())
print(combined['judge'].value_counts())

## Step 8 — Analysis (tables, confidence intervals, figures)

Reads the merged file and writes all analysis outputs into a `results/` subfolder inside the combined Drive folder (the script adds that subfolder itself).

In [ ]:
!python 06_analysis.py --results-csv $RUN_DIR/combined/results.csv --out-dir $RUN_DIR/combined --bootstrap 1000 --ci 0.95

## Step 9 — Verify outputs

In [ ]:
!ls -la $RUN_DIR/combined/results
import json
try:
    s = json.load(open(f'{RUN_DIR}/combined/results/summary.json'))
    print(json.dumps(s, indent=2)[:2000])
except Exception as e:
    print('summary.json not found yet:', e)

## Troubleshooting & notes

- **A model cell crashes:** re-run the same cell. It prints `[resume] loaded N rows / M completed prompts` and continues; finished prompts are not recomputed. Finished models are untouched.
- **Force a clean re-run of a model:** delete that model's Drive folder (or add `--no-resume`) before re-running.
- **Out-of-memory on the 13B judge:** add `--harmbench-cls-size small` (Mistral-7B) to that model's command.
- **Runtime fully reset:** re-run Steps 1–4, then re-run the model cells — each resumes from its Drive checkpoint, so completed prompts cost nothing.
- **Registration check:** open any `$RUN_DIR/<model>/run_manifest.json` and confirm its `timestamp` is later than your GitHub push / arXiv time.
- **Benchmark sizes:** this uses `--n 200`, so XSTest is sampled to 200. Your pre-registration lists XSTest-250 (and OR-Bench-Hard). For full XSTest-250 use `--n 250`; to add OR-Bench-Hard add `--enable-orbench` (much longer run). Confirm against your registered protocol before deciding.
- **The 3 models from the earlier run are NOT reusable here:** that run scored only keyword + Llama-Guard and did not save raw completions, so the HarmBench judge-of-record cannot be added without regenerating. This notebook regenerates all four with all three judges — the earlier partial compute is sunk, but resume guarantees no further waste.
- **HF token:** rotate it — the previous one was exposed.